# WaveDecompNet reproduction, Phase 1 (E1): train, evaluate, compare, scorecard

Kaggle GPU notebook. It contains no report text. It only produces numbers, CSV files and figures.

**Rules built into this notebook**
- The training data are already built. Nothing here rebuilds or edits them.
- Model, loss, optimiser, schedule, split, hyperparameters and seeds are those of the patched `step3_training_1D.py`, which is embedded unchanged and called by `subprocess`.
- Paper numbers are never filled in by the notebook. You paste them into `PAPER_VALUES` in the configuration cell.

**How the notebook is organised**

| Part | Switch | What it does | Needs |
|---|---|---|---|
| 0 | always on (setup); `RUN_DATA_SCAN` for the full scan | environment, repo, data checks, lab log | the data dataset attached |
| A | `RUN_PART_A` | authors' pretrained models on the Hawaii example files | Internet (git clone) |
| B | `RUN_PART_B_SMOKE`, `RUN_PART_B_FULL` | train one bottleneck and one seed per session | GPU |
| C | `RUN_PART_C` | evaluate all finished runs and the pretrained models on the same test set | the run datasets attached |
| D | `RUN_PART_D` | real continuous data (step6 and step7a logic) | one trained or pretrained model |
| E | `RUN_PART_E` | scorecard, deviation log, per-run config files | results of Part C |

Always run the first four code cells (configuration, setup, embedded scripts, data checks). Then switch on only the part you want.

**Moving results between Kaggle sessions**
1. Train one bottleneck and one seed per session (Part B). The run folder is copied to `/kaggle/working/wdn_runs/Branch_Encoder_Decoder_<bottleneck>_warmup[_seed<seed>]/`.
2. Save the notebook version (Save Version, then Save and Run All) so the output is kept. From the version's Output tab, create a new Kaggle dataset from the output (menu names may differ; check the current Kaggle UI). Keep only the `wdn_runs` folder if the UI lets you choose files.
3. In the evaluation session attach the data dataset, every run dataset, and nothing else. Part C finds runs with `glob('**/Branch_Encoder_Decoder_*/run_info.json')`, so the exact mount path does not matter.
4. Expected layout of one run folder: `..._Model.pth`, `..._Training_history.hdf5`, `..._Dataset_split.hdf5`, `run_info.json`, `..._Training_history.png`, and `config_<run>.json` (written by this notebook). Expected size: a few MB per run (the pretrained files of the authors are 0.7 MB to 5.2 MB). The notebook output folder as a whole stays well below the 20 GB limit.


In [1]:
# ======================= SWITCHES: set True only for the parts you want in THIS session =======================
RUN_DATA_SCAN    = False   # Part 0: scan all of X_train / Y_train for non-finite values (reads 5.4 GB)
RUN_PART_A       = False   # authors' pretrained models on the Hawaii example files
RUN_PART_B_SMOKE = True   # 2-epoch smoke run on 5000 samples, prints the projected time for 100 epochs
RUN_PART_B_FULL  = True   # full training run of BOTTLENECK with SEED
RUN_PART_C       = False   # evaluation, tables, figures
RUN_PART_D       = False   # real continuous data
RUN_PART_E       = False   # scorecard, deviation log, config files

# ======================= Part B: what to train in this session =======================
BOTTLENECKS     = ["None", "Linear", "LSTM", "attention", "Transformer"]
BOTTLENECK      = "Linear"        # one of BOTTLENECKS
SEED            = 99            # torch seed. 99 is the repo default
EPOCHS          = 100           # the repo value; the smoke test uses its own epochs
PLANNED_SEEDS   = [99, 13, 20]  # 99 = repo default; 13 and 20 are the other seeds named in the workflow file
SESSION_LIMIT_H = 12.0          # CHECK your Kaggle account page and edit (hours per GPU session)
SAFETY_FACTOR   = 0.9           # stop if the projected time exceeds SAFETY_FACTOR * SESSION_LIMIT_H
FORCE_FULL_RUN  = False         # True = start the full run even without a passing smoke projection

# ======================= Part C: evaluation settings =======================
EPOCHS_REQUIRED     = 100       # only runs with this many epochs and no --subset are evaluated
C_RECOMPUTE         = False     # False = reuse results/per_sample_*.csv.gz if they exist
EV_CLIP             = -1.0      # lower clip for the "clipped mean" EV (see the note in Part C)
BANDPASS_HZ         = (0.5, 4.9)  # baseline band. Lower edge: highpass 0.5 Hz in the repo example notebook.
                                  # Upper edge: just below the 5 Hz Nyquist of the 10 Hz data. CONFIRM against the paper.
N_BOOT              = 1000      # bootstrap resamples
SNR_BINS_PER_DECADE = 4         # log-spaced bins of the linear SNR between 0.1 and 100
FAIL_FRACTION       = 0.02      # worst 2 % of test samples are saved as failure cases
FAILURE_MODEL_ID    = None      # None = first LSTM run found, otherwise the first run of ours
BOOT_SEED           = 12345
EXAMPLE_SEED        = 7

# ======================= Part D: real data =======================
D_MODEL_SOURCE  = "ours"        # "ours" (needs a finished run attached) or "pretrained"
D_BOTTLENECK    = "LSTM"
D_SEED          = 99
D_STATION_FILE  = "HV.HAT.20210807-20210808.mseed"   # or "IU.POHA.20210807-20210808.mseed"
D_PREPROCESS    = "step6"       # "step6": downsample_series, no filter.  "example_nb": decimate + highpass 0.5 Hz
D_STALTA        = dict(short_term=2, long_term=60, trigger_on=6, trigger_off=2, threshold_coincidence=1)  # step7a values
D_SCALE         = 1e5           # step7a scaling_factor_waveform

# ======================= Paper values: paste them here. None = unknown. Nothing is filled in for you. =======================
PAPER_VALUES = {
    "ev_earthquake": {b: None for b in BOTTLENECKS},   # e.g. {"None": 0.95, ...}
    "ev_noise":      {b: None for b in BOTTLENECKS},
    "test_loss":     {b: None for b in BOTTLENECKS},
    "n_params":      {b: None for b in BOTTLENECKS},
}


In [2]:
import os, sys, glob, json, time, shutil, subprocess, platform, datetime, re, traceback, hashlib
from pathlib import Path

ON_KAGGLE = os.path.exists("/kaggle/input")
BASE = Path("/kaggle/working") if ON_KAGGLE else Path.cwd()
RESULTS, LOGS, FIGS, RUNS_OUT = BASE/"results", BASE/"logs", BASE/"figures", BASE/"wdn_runs"
for d in (RESULTS, LOGS, FIGS, RUNS_OUT):
    d.mkdir(parents=True, exist_ok=True)
CODE_DIR = BASE/"WaveDecompNet-paper"
MODEL_DIR = BASE/"Model_and_datasets_1D_all_snr_40_unshuffled_equal_epoch100"
REPO_URL = "https://github.com/yinjiuxun/WaveDecompNet-paper"
EXPECTED_COMMIT = "afe2536cf29ead8b4d8d5c50ddf85cc1dd90ddd1"

def sh(cmd, cwd=None):
    r = subprocess.run(cmd, cwd=cwd, capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError(f"{' '.join(map(str, cmd))}\n{r.stdout}\n{r.stderr}")
    return r.stdout.strip()

# ---- environment printout
print("python", platform.python_version())
import numpy as np
print("numpy", np.__version__)
try:
    import torch
    print("torch", torch.__version__, "| cuda available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0), "| memory GB:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))
except Exception as e:
    print("torch import failed:", e)
try:
    import psutil
    print("RAM GB total / available:", round(psutil.virtual_memory().total / 1e9, 1), "/", round(psutil.virtual_memory().available / 1e9, 1))
except Exception:
    pass
print("disk free GB (working dir):", round(shutil.disk_usage(BASE).free / 1e9, 1))

# ---- packages used later: installed only if missing
def ensure(pip_name, module=None):
    try:
        __import__(module or pip_name)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pip_name], check=True)
ensure("obspy"); ensure("torchinfo"); ensure("h5py"); ensure("pandas"); ensure("scikit-learn", "sklearn")

# ---- repo at the recorded commit
if not (CODE_DIR/".git").exists():
    sh(["git", "clone", "--depth", "1", REPO_URL, str(CODE_DIR)])
GIT_HASH = sh(["git", "-C", str(CODE_DIR), "rev-parse", "HEAD"])
if GIT_HASH != EXPECTED_COMMIT:
    sh(["git", "-C", str(CODE_DIR), "fetch", "--depth", "1", "origin", EXPECTED_COMMIT])
    sh(["git", "-C", str(CODE_DIR), "checkout", EXPECTED_COMMIT])
    GIT_HASH = sh(["git", "-C", str(CODE_DIR), "rev-parse", "HEAD"])
assert GIT_HASH == EXPECTED_COMMIT, f"repo commit {GIT_HASH} differs from {EXPECTED_COMMIT}"
print("repo commit:", GIT_HASH)

# ---- lab log
def log_entry(part, command, outcome, seed=None):
    p = LOGS/"lab_log.md"
    if not p.exists():
        p.write_text("# Lab log\n\n| date (UTC) | part | command | commit | seed | outcome |\n|---|---|---|---|---|---|\n")
    now = datetime.datetime.now(datetime.timezone.utc).strftime("%Y-%m-%d %H:%M")
    with open(p, "a") as f:
        f.write(f"| {now} | {part} | `{command}` | {GIT_HASH[:10]} | {seed if seed is not None else ''} | {outcome} |\n")

def find_files(name):
    return sorted(glob.glob(f"/kaggle/input/**/{name}", recursive=True))

def find_one(name, required=True):
    hits = find_files(name)
    if not hits and required:
        raise FileNotFoundError(f"{name} not found under /kaggle/input. Attach the dataset that contains it.")
    if len(hits) > 1:
        print(f"WARNING: {name} found {len(hits)} times, using {hits[0]}")
    return Path(hits[0]) if hits else None


python 3.13.16
numpy 2.1.3
torch 2.11.0+cu130 | cuda available: True
GPU: Tesla T4 | memory GB: 15.6
RAM GB total / available: 33.7 / 32.0
disk free GB (working dir): 20.9
repo commit: afe2536cf29ead8b4d8d5c50ddf85cc1dd90ddd1


In [3]:
# Embedded, UNCHANGED copies of the patched wdn_code files (config.py and step3_training_1D.py).
# They are written next to the repo files so `step3` imports them exactly as in wdn_code.
CONFIG_PY = r'''"""Central configuration for the WaveDecompNet replication (works locally and on Kaggle).

Every other script imports this file, so paths and seeds live in exactly one place.
Override the STEAD location with the environment variable WDN_STEAD_DIR if needed.
"""
import glob
import os

ON_KAGGLE = os.path.exists("/kaggle/input")


def _find_stead(name):
    env = os.environ.get("WDN_STEAD_DIR")
    if env:
        return os.path.join(env, name)
    if ON_KAGGLE:
        hits = glob.glob(f"/kaggle/input/**/{name}", recursive=True)
        if hits:
            return hits[0]
    return os.path.join(os.getcwd(), "stead", name)  # local fallback


# ---------------------------------------------------------------- paths
WORK_DIR = "/kaggle/working" if ON_KAGGLE else os.getcwd()

STEAD_HDF5 = _find_stead("merged.hdf5")
STEAD_CSV = _find_stead("merged.csv")

DATASET_DIR = os.path.join(WORK_DIR, "training_datasets")
POHA_DIR = os.path.join(WORK_DIR, "continuous_waveforms")
POHA_MSEED = os.path.join(POHA_DIR, "IU.POHA.00.20210601-20210801.mseed")

STEAD_MIX_H5 = os.path.join(DATASET_DIR, "training_datasets_STEAD_waveform_snr_40.hdf5")
POHA_MIX_H5 = os.path.join(DATASET_DIR, "training_datasets_STEAD_plus_POHA_snr_40_unshuffled.hdf5")
# paper-faithful name (the file is actually shuffled WITH replacement, see step2d)
FINAL_DATASET = os.path.join(DATASET_DIR, "training_datasets_all_snr_40_unshuffled.hdf5")
FINAL_DATASET_PERMUTED = os.path.join(DATASET_DIR, "training_datasets_all_snr_40_permuted.hdf5")

MODEL_DIR = os.path.join(WORK_DIR, "Model_and_datasets_1D_all_snr_40_unshuffled_equal_epoch100")

# ---------------------------------------------------------------- data recipe (from the repo)
SNR_MIN_DB = 40
N_EVENTS = 100000                 # step2b: number of STEAD mixtures
F_DOWNSAMPLE = 10                 # Hz
N_APPEND = 1000                   # chunk size when writing HDF5
STEAD_SEEDS = dict(earthquake=101, noise=102, shift=103, snr=104)   # step2b
POHA_SEEDS = dict(earthquake=111, noise=112, shift=113, snr=114)    # step2c
POHA_AMPLITUDE_THRESHOLD = 6      # step2c: drop samples above 6 x median amplitude
POHA_EXPECTED_FS = 40.0           # step2c calls decimate(4) -> 10 Hz, so it assumes 40 Hz

# step2c in the original code writes only full chunks of N_APPEND, so the last
# (N mod 1000) windows are silently dropped. False = keep that behaviour (fidelity).
KEEP_REMAINDER = False

# step2d in the original code shuffles with np.random.choice(N, N): WITH replacement,
# no seed. True = same behaviour but seeded and logged; False = a proper permutation.
SHUFFLE_WITH_REPLACEMENT = True
SHUFFLE_SEED = 0

# ---------------------------------------------------------------- training (step3)
BATCH_SIZE = 128
EPOCHS = 100
LR = 1e-3
MIN_EPOCHS = 30
PATIENCE = None                   # None = no early stopping (always 100 epochs)
TORCH_SEED = 99
SPLIT_SEED_1 = 13
SPLIT_SEED_2 = 20
TRAIN_SIZE = 0.6
TEST_SIZE = 0.5                   # of the remaining 40 % -> 20 % val, 20 % test
'''

STEP3_PY = r'''"""step3 (patched): train ONE bottleneck per run (needed for Kaggle's session limits).

Same model, loss, optimiser, LR schedule, split (60/20/20, seeds 13 and 20) and seeds as the repo.
Changes: command-line arguments, paths from config.py, no `tabnanny` import, no `verbose=` in
LambdaLR (removed in newer PyTorch), figure saved headless, run info saved as JSON.

Quick smoke test first (a few minutes):
    python step3_training_1D.py --bottleneck None --epochs 2 --subset 5000 --tag _smoke
Full run:
    python step3_training_1D.py --bottleneck LSTM
"""
import argparse
import copy
import json
import os
import random
import time

import h5py
import matplotlib
matplotlib.use("Agg")
import numpy as np
import torch
from matplotlib import pyplot as plt
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader

import config as C
from autoencoder_1D_models_torch import SeismogramDecoder, SeismogramEncoder, SeisSeparator
from torch_tools import WaveformDataset, training_loop_branches, try_gpu

BOTTLENECKS = ["None", "Linear", "LSTM", "attention", "Transformer"]


def make_bottleneck(name):
    if name == "None":
        return None
    if name == "Linear":
        return torch.nn.Linear(64, 64, dtype=torch.float64)
    if name == "LSTM":
        return torch.nn.LSTM(64, 32, 2, bidirectional=True, batch_first=True, dtype=torch.float64)
    if name == "attention":
        return torch.nn.MultiheadAttention(64, 4, 0.2, batch_first=True, dtype=torch.float64)
    if name == "Transformer":
        layer = torch.nn.TransformerEncoderLayer(d_model=64, nhead=4, dtype=torch.float64)
        return torch.nn.TransformerEncoder(layer, num_layers=1)
    raise NameError("bottleneck type has not been defined!")


def main():
    p = argparse.ArgumentParser()
    p.add_argument("--bottleneck", required=True, choices=BOTTLENECKS)
    p.add_argument("--data", default=C.FINAL_DATASET)
    p.add_argument("--seed", type=int, default=C.TORCH_SEED)
    p.add_argument("--epochs", type=int, default=C.EPOCHS)
    p.add_argument("--subset", type=int, default=None, help="use only the first N samples (smoke test)")
    p.add_argument("--tag", default=None, help="suffix of the output folder")
    args = p.parse_args()

    tag = args.tag if args.tag is not None else ("_warmup" if args.seed == C.TORCH_SEED else f"_warmup_seed{args.seed}")
    model_structure = "Branch_Encoder_Decoder"
    model_name = f"{model_structure}_{args.bottleneck}"
    out_dir = os.path.join(C.MODEL_DIR, model_name + tag)
    os.makedirs(out_dir, exist_ok=True)
    device = try_gpu()
    print("device:", device, torch.cuda.get_device_name(0) if device.type == "cuda" else "")

    # ---- data
    print("#" * 12 + " Loading data " + "#" * 12)
    with h5py.File(args.data, "r") as f:
        n = f["X_train"].shape[0] if args.subset is None else min(args.subset, f["X_train"].shape[0])
        X_train, Y_train = f["X_train"][:n], f["Y_train"][:n]
    print("data:", X_train.shape, X_train.dtype)

    X_training, X_test, Y_training, Y_test = train_test_split(
        X_train, Y_train, train_size=C.TRAIN_SIZE, random_state=C.SPLIT_SEED_1)
    X_validate, X_test, Y_validate, Y_test = train_test_split(
        X_test, Y_test, test_size=C.TEST_SIZE, random_state=C.SPLIT_SEED_2)
    del X_train, Y_train, X_test, Y_test

    # ---- seeds (same as the repo, torch seed configurable for multi-seed runs)
    torch.manual_seed(args.seed)
    random.seed(0)
    np.random.seed(20)
    torch.backends.cudnn.benchmark = False

    training_data = WaveformDataset(X_training, Y_training)
    validate_data = WaveformDataset(X_validate, Y_validate)

    # ---- model
    bottleneck = make_bottleneck(args.bottleneck)
    encoder = SeismogramEncoder()
    decoder_earthquake = SeismogramDecoder(bottleneck=copy.deepcopy(bottleneck))
    decoder_noise = SeismogramDecoder(bottleneck=copy.deepcopy(bottleneck))
    model = SeisSeparator(model_name, encoder, decoder_earthquake, decoder_noise).to(device=device)
    n_params = int(sum(p.numel() for p in model.parameters()))
    print(f"{model_name}: {n_params} parameters")

    batch_size, lr = C.BATCH_SIZE, C.LR
    loss_fn = torch.nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    LR_func = lambda epoch: (epoch + 1) / 11 if (epoch <= 10) else (0.95 ** epoch if (epoch < 50) else 0.95 ** 50)
    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=LR_func)

    train_iter = DataLoader(training_data, batch_size=batch_size, shuffle=True)
    validate_iter = DataLoader(validate_data, batch_size=batch_size, shuffle=False)

    # ---- train
    print("#" * 12 + " training model " + model_name + " " + "#" * 12)
    t0 = time.time()
    model, avg_train_losses, avg_valid_losses, partial_loss = training_loop_branches(
        train_iter, validate_iter, model, loss_fn, optimizer, scheduler,
        epochs=args.epochs, patience=C.PATIENCE, device=device, minimum_epochs=C.MIN_EPOCHS)
    runtime = time.time() - t0
    print(f"Training is done! {runtime / 60:.1f} min")

    # ---- save
    torch.save(model, os.path.join(out_dir, f"{model_name}_Model.pth"))
    with h5py.File(os.path.join(out_dir, f"{model_name}_Training_history.hdf5"), "w") as f:
        f.create_dataset("loss", data=avg_train_losses)
        f.create_dataset("val_loss", data=avg_valid_losses)
        f.create_dataset("earthquake_loss", data=partial_loss[0])
        f.create_dataset("earthquake_val_loss", data=partial_loss[1])
        f.create_dataset("noise_loss", data=partial_loss[2])
        f.create_dataset("noise_val_loss", data=partial_loss[3])
    with h5py.File(os.path.join(out_dir, f"{model_name}_Dataset_split.hdf5"), "w") as f:
        f.attrs["model_name"] = model_name
        f.attrs["train_size"] = C.TRAIN_SIZE
        f.attrs["test_size"] = C.TEST_SIZE
        f.attrs["rand_seed1"] = C.SPLIT_SEED_1
        f.attrs["rand_seed2"] = C.SPLIT_SEED_2
    with open(os.path.join(out_dir, "run_info.json"), "w") as f:
        json.dump(dict(model_name=model_name, bottleneck=args.bottleneck, torch_seed=args.seed,
                       epochs=args.epochs, subset=args.subset, data=args.data,
                       n_train=len(training_data), n_val=len(validate_data), n_params=n_params,
                       runtime_s=runtime, device=str(device),
                       gpu=torch.cuda.get_device_name(0) if device.type == "cuda" else None,
                       torch=torch.__version__, final_train_loss=float(avg_train_losses[-1]),
                       final_val_loss=float(avg_valid_losses[-1])), f, indent=2)

    plt.figure()
    plt.plot(avg_train_losses, "o", label="loss")
    plt.plot(avg_valid_losses, "-", label="Validation loss")
    for k, (lab, mk) in enumerate(zip(["earthquake train loss", "earthquake valid loss",
                                       "noise train loss", "noise valid loss"], ["o", "", "o", ""])):
        plt.plot(partial_loss[k], marker=mk, label=lab)
    plt.yscale("log")
    plt.legend()
    plt.title(model_name)
    plt.savefig(os.path.join(out_dir, f"{model_name}_Training_history.png"))
    plt.close()
    print("saved to", out_dir)


if __name__ == "__main__":
    main()
'''


(CODE_DIR/"config.py").write_text(CONFIG_PY)
(CODE_DIR/"step3_training_1D.py").write_text(STEP3_PY)
SCRIPT_SHA = {n: hashlib.sha256((CODE_DIR/n).read_bytes()).hexdigest()[:16] for n in ("config.py", "step3_training_1D.py")}
print("embedded scripts written, sha256[:16]:", SCRIPT_SHA)

# one-line NumPy 2 fix from PATCH_NOTES (np.Inf -> np.inf); idempotent
tt = CODE_DIR/"torch_tools.py"
txt = tt.read_text()
if "np.Inf" in txt:
    tt.write_text(txt.replace("np.Inf", "np.inf")); print("patched np.Inf -> np.inf in torch_tools.py")
else:
    print("torch_tools.py: no np.Inf left")
assert "np.Inf" not in tt.read_text()
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))


embedded scripts written, sha256[:16]: {'config.py': '146bcb7aa227c74e', 'step3_training_1D.py': '1e186284c55e79d2'}
patched np.Inf -> np.inf in torch_tools.py


In [4]:
import h5py
TRAIN_NAME = "training_datasets_all_snr_40_unshuffled.hdf5"
POHA_NAME  = "training_datasets_STEAD_plus_POHA_snr_40_unshuffled.hdf5"
STEAD_NAME = "training_datasets_STEAD_waveform_snr_40.hdf5"
TRAIN_PATH = find_one(TRAIN_NAME, required=False)
POHA_PATH  = find_one(POHA_NAME, required=False)
STEAD_PATH = find_one(STEAD_NAME, required=False)
print("training file:", TRAIN_PATH, "\nPOHA mixtures:", POHA_PATH, "\nSTEAD mixtures:", STEAD_PATH)
DATA_FACTS = {}
if TRAIN_PATH is None:
    print("STOP: the training file is not attached. Parts B and C need it (Parts A, D do not).")
else:
    with h5py.File(TRAIN_PATH, "r") as f:
        for k in f:
            print(f"  {k:14s} {f[k].shape} {f[k].dtype}")
        print("  attrs:", {k: (v.item() if hasattr(v, "item") else v) for k, v in f.attrs.items()})
        assert f["X_train"].shape == (187000, 600, 3) and f["Y_train"].shape == (187000, 600, 3), "unexpected data shape"
        t = f["time_new"][:]
        assert len(t) == 600 and abs((t[1] - t[0]) - 0.1) < 1e-9, "not 600 samples at 10 Hz"
        src = f["source_index"][:]
        N_POHA = int(f.attrs.get("n_poha", 87000)); N_STEAD = int(f.attrs.get("n_stead", 100000))
        assert N_POHA == 87000 and N_STEAD == 100000, (N_POHA, N_STEAD)
        DATA_FACTS = dict(n_rows=int(src.shape[0]), n_poha=N_POHA, n_stead=N_STEAD, n_unique_source_index=int(len(np.unique(src))),
                          pct_unique=float(100 * len(np.unique(src)) / src.shape[0]),
                          attrs={k: (v.item() if hasattr(v, "item") else str(v)) for k, v in f.attrs.items()})
    print("shape (187000, 600, 3), 10 Hz, 600 samples: OK")
    print(f"unique source_index: {DATA_FACTS['n_unique_source_index']} of {DATA_FACTS['n_rows']} ({DATA_FACTS['pct_unique']:.1f} %)")
    for nm, p, n in (("POHA", POHA_PATH, N_POHA), ("STEAD", STEAD_PATH, N_STEAD)):
        if p is None:
            print(f"NOTE: {nm} mixture file not attached: per-sample SNR/shift metadata and the leakage audit will be unavailable.")
            continue
        with h5py.File(p, "r") as f:
            print(nm, {k: f[k].shape for k in f})
            assert f["X_train"].shape[0] == n, f"{nm} file has {f['X_train'].shape[0]} rows, expected {n}"
json.dump(DATA_FACTS, open(RESULTS/"data_facts.json", "w"), indent=2)
log_entry("0 setup", "setup + data checks", f"commit ok; data facts saved; scripts {SCRIPT_SHA}")


training file: /kaggle/input/datasets/prabhanshu22/wdn-training-data/training_datasets_all_snr_40_unshuffled.hdf5 
POHA mixtures: /kaggle/input/datasets/prabhanshu22/wdn-training-data/training_datasets_STEAD_plus_POHA_snr_40_unshuffled.hdf5 
STEAD mixtures: /kaggle/input/datasets/prabhanshu22/wdn-training-data/training_datasets_STEAD_waveform_snr_40.hdf5
  X_train        (187000, 600, 3) float64
  Y_train        (187000, 600, 3) float64
  source_index   (187000,) int64
  time_new       (600,) float64
  attrs: {'n_poha': 87000, 'n_stead': 100000, 'n_unique': 118217, 'shuffle_seed': 0, 'with_replacement': True}
shape (187000, 600, 3), 10 Hz, 600 samples: OK
unique source_index: 118217 of 187000 (63.2 %)
POHA {'X_train': (87000, 600, 3), 'Y_train': (87000, 600, 3), 'earthquake_name': (87000,), 'noise_window_index': (87000,), 'quake_energy_frac': (87000,), 'shift': (87000,), 'snr': (87000,), 'time': (600,)}
STEAD {'X_train': (100000, 600, 3), 'Y_train': (100000, 600, 3), 'earthquake_name':

## Part 0 (optional): full-file scan
Switch `RUN_DATA_SCAN`. Reads the whole 5.4 GB file in chunks; checks for non-finite values and prints the mean and std of X per source.

In [5]:
if RUN_DATA_SCAN and TRAIN_PATH is not None:
    t0 = time.time()
    bad_x = bad_y = 0
    acc = {"POHA": [0, 0.0, 0.0], "STEAD": [0, 0.0, 0.0]}   # count, sum, sum of squares of X
    with h5py.File(TRAIN_PATH, "r") as f:
        n = f["X_train"].shape[0]; src = f["source_index"][:]
        for lo in range(0, n, 5000):
            hi = min(lo + 5000, n)
            x = f["X_train"][lo:hi]; y = f["Y_train"][lo:hi]
            bad_x += int((~np.isfinite(x)).sum()); bad_y += int((~np.isfinite(y)).sum())
            isp = src[lo:hi] < N_POHA
            for nm, m in (("POHA", isp), ("STEAD", ~isp)):
                if m.any():
                    xs = x[m]; acc[nm][0] += xs.size; acc[nm][1] += float(xs.sum()); acc[nm][2] += float((xs ** 2).sum())
    scan = {"non_finite_X": bad_x, "non_finite_Y": bad_y, "seconds": round(time.time() - t0, 1)}
    for nm, (c, s, s2) in acc.items():
        mean = s / c; scan[f"X_mean_{nm}"] = mean; scan[f"X_std_{nm}"] = float(np.sqrt(s2 / c - mean ** 2))
    print(scan)
    print("OK: no non-finite values" if bad_x == 0 and bad_y == 0 else "STOP: non-finite values found, do not train yet")
    json.dump(scan, open(RESULTS/"data_scan.json", "w"), indent=2)
    log_entry("0 scan", "full-file NaN/Inf scan", f"non-finite X={bad_x}, Y={bad_y}")
else:
    print("data scan skipped (RUN_DATA_SCAN = False)")


data scan skipped (RUN_DATA_SCAN = False)


## Helpers (always run, cheap)

In [6]:
import pandas as pd

def read_csv_keep(path):
    """pandas treats the string 'None' (our bottleneck name) as missing by default; keep it."""
    return pd.read_csv(path, keep_default_na=False, na_values=[""])

def save_fig(fig, name):
    for ext in ("pdf", "png"):
        fig.savefig(FIGS/f"{name}.{ext}", bbox_inches="tight", dpi=200)

def prepare_torch():
    """Import torch and the repo modules; disable the eval fast path so eval == the training-time math."""
    import importlib, torch
    if str(CODE_DIR) not in sys.path:
        sys.path.insert(0, str(CODE_DIR))
    mods = {n: importlib.import_module(n) for n in ("autoencoder_1D_models_torch", "torch_tools", "utilities")}
    try:
        torch.backends.mha.set_fastpath_enabled(False)
    except Exception:
        pass
    return torch, mods

COMPAT_DEFAULTS = {"norm_first": False, "activation_relu_or_gelu": 1, "enable_nested_tensor": False,
                   "use_nested_tensor": False, "mask_check": True}
COMPAT_LOG = []

def load_model(path, device):
    """Load a whole pickled model (torch.save(model)). Needs autoencoder_1D_models_torch importable."""
    torch, _ = prepare_torch()
    try:
        m = torch.load(str(path), map_location=device, weights_only=False)
    except TypeError:
        m = torch.load(str(path), map_location=device)
    return m.to(device).double().eval()

def check_forward(model, device):
    """Dummy forward pass. Optional compat shim ONLY if an old pickle lacks an attribute newer torch expects
    (defaults = what the constructor sets). Every use is recorded in COMPAT_LOG and goes into the deviation log."""
    torch, _ = prepare_torch()
    x = torch.randn(2, 3, 600, dtype=torch.float64, device=device)
    for _ in range(8):
        try:
            with torch.no_grad():
                o = model(x)
            assert len(o) == 2 and tuple(o[0].shape) == (2, 3, 600) and tuple(o[1].shape) == (2, 3, 600), "unexpected output shapes"
            return True, None
        except AttributeError as e:
            name = (re.search(r"has no attribute '(\w+)'", str(e)) or [None, None])[1]
            if name not in COMPAT_DEFAULTS:
                return False, traceback.format_exc()
            for mod in model.modules():
                if type(mod).__name__ in ("TransformerEncoderLayer", "TransformerEncoder") and not hasattr(mod, name):
                    setattr(mod, name, COMPAT_DEFAULTS[name])
            COMPAT_LOG.append(name)
        except Exception:
            return False, traceback.format_exc()
    return False, "compat shim did not converge: " + ", ".join(COMPAT_LOG)

def predict(model, Xt, device, bs=256):
    """Xt: (N, 3, 600) float64 -> (earthquake, noise), both (N, 3, 600)."""
    torch, _ = prepare_torch()
    pe = np.empty_like(Xt); pn = np.empty_like(Xt)
    with torch.no_grad():
        for i in range(0, len(Xt), bs):
            xb = torch.from_numpy(Xt[i:i + bs]).to(device)
            o1, o2 = model(xb)
            pe[i:i + bs] = o1.cpu().numpy(); pn[i:i + bs] = o2.cpu().numpy()
    return pe, pn

def get_device():
    torch, _ = prepare_torch()
    return torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

# ------------------------------------------------------------------ runs
def run_folder_name(bottleneck, seed, tag=None):
    tag = tag if tag is not None else ("_warmup" if seed == 99 else f"_warmup_seed{seed}")   # same rule as step3
    return f"Branch_Encoder_Decoder_{bottleneck}{tag}"

def discover_runs(epochs_required):
    """All finished full runs (no --subset, required epochs) under /kaggle/input and the working dir."""
    found = {}
    roots = ["/kaggle/input", str(BASE)] if ON_KAGGLE else [str(BASE)]
    for root in roots:
        for p in sorted(glob.glob(f"{root}/**/Branch_Encoder_Decoder_*/run_info.json", recursive=True)):
            try:
                info = json.load(open(p))
            except Exception:
                continue
            if info.get("subset") is not None or info.get("epochs") != epochs_required:
                continue
            d = Path(p).parent
            mf = d/f"{info['model_name']}_Model.pth"
            if not mf.exists():
                continue
            key = (info["bottleneck"], int(info["torch_seed"]))
            if key not in found:
                found[key] = dict(dir=d, model_file=mf, info=info)
    return found

def update_run_table():
    found = discover_runs(EPOCHS_REQUIRED)
    keys = [(b, s) for b in BOTTLENECKS for s in PLANNED_SEEDS] + sorted(k for k in found if k[1] not in PLANNED_SEEDS)
    rows = []
    for b, s in keys:
        r = found.get((b, s)); i = r["info"] if r else {}
        rows.append(dict(bottleneck=b, seed=s, status="done" if r else "planned", run_dir=str(r["dir"]) if r else "",
                         n_params=i.get("n_params"), runtime_s=i.get("runtime_s"), gpu=i.get("gpu"),
                         final_train_loss=i.get("final_train_loss"), final_val_loss=i.get("final_val_loss")))
    df = pd.DataFrame(rows); df.to_csv(RESULTS/"run_table.csv", index=False)
    return df

def write_run_config(run):
    """config_<run>.json: git hash, seed, bottleneck, dataset, package versions, date."""
    i = run["info"]; out = Path(run["dir"])/f"config_{i['model_name']}.json"
    cfg = dict(git_hash=GIT_HASH, seed=i["torch_seed"], bottleneck=i["bottleneck"], dataset=str(i.get("data")),
               torch=i.get("torch"), gpu=i.get("gpu"), numpy=np.__version__, python=platform.python_version(),
               script_sha=SCRIPT_SHA, date_utc=datetime.datetime.now(datetime.timezone.utc).isoformat())
    try:
        out.write_text(json.dumps(cfg, indent=2))
    except OSError:                       # read-only input dataset: keep a copy in results/
        (RESULTS/out.name).write_text(json.dumps(cfg, indent=2))


In [7]:
from sklearn.model_selection import train_test_split

def ev_score(y, p, axis=-1):
    """Explained variance exactly as sklearn.explained_variance_score(multioutput='raw_values') (the definition used in step5),
    vectorised: one value per trace and component, time on `axis`."""
    d = y - p
    num = np.mean((d - d.mean(axis, keepdims=True)) ** 2, axis=axis)
    den = np.mean((y - y.mean(axis, keepdims=True)) ** 2, axis=axis)
    out = np.ones_like(num)
    ok = (num != 0) & (den != 0)
    out[ok] = 1 - num[ok] / den[ok]
    out[(num != 0) & (den == 0)] = 0.0
    return out

def step5_snr_db(signal, noise, axis=-1):
    """signal_to_noise_ratio() of step5, in dB, per component."""
    s0 = np.sum(signal ** 2, axis=axis) / (np.sum(noise ** 2, axis=axis) + 1e-6)
    return 10 * np.log10(s0 + 1e-6)

def pearson_rows(a, b):
    a = a.reshape(len(a), -1); b = b.reshape(len(b), -1)
    a = a - a.mean(1, keepdims=True); b = b - b.mean(1, keepdims=True)
    den = np.sqrt((a ** 2).sum(1) * (b ** 2).sum(1))
    with np.errstate(divide="ignore", invalid="ignore"):
        return np.where(den > 0, (a * b).sum(1) / den, np.nan)

def metrics_chunked(Xt, Yt, pe, pn, chunk=4000):
    """Per-sample metrics. Xt, Yt, pe, pn: (N, 3, 600). Earthquake truth = Yt, noise truth = Xt - Yt (as in training)."""
    cols = {k: [] for k in ("ev_eq", "ev_nz", "snr5", "mse_eq", "mse_noise", "corr")}
    for i in range(0, len(Xt), chunk):
        x, y, a, b = Xt[i:i + chunk], Yt[i:i + chunk], pe[i:i + chunk], pn[i:i + chunk]
        tn = x - y
        cols["ev_eq"].append(ev_score(y, a)); cols["ev_nz"].append(ev_score(tn, b))
        cols["snr5"].append(step5_snr_db(y, tn))
        cols["mse_eq"].append(((y - a) ** 2).mean((1, 2))); cols["mse_noise"].append(((tn - b) ** 2).mean((1, 2)))
        cols["corr"].append(pearson_rows(y, a))
    return {k: np.concatenate(v) for k, v in cols.items()}

def per_sample_frame(meta, m):
    df = meta[["row", "file_row", "source_index", "source", "snr", "shift", "quake_energy_frac"]].copy()
    for c in range(3):
        df[f"ev_eq_c{c}"] = m["ev_eq"][:, c]
    df["ev_eq_mean_clipped"] = np.clip(m["ev_eq"], EV_CLIP, 1).mean(1)
    for c in range(3):
        df[f"ev_noise_c{c}"] = m["ev_nz"][:, c]
    df["ev_noise_mean_clipped"] = np.clip(m["ev_nz"], EV_CLIP, 1).mean(1)
    df["mse_eq"] = m["mse_eq"]; df["mse_noise"] = m["mse_noise"]; df["mse_total"] = m["mse_eq"] + m["mse_noise"]
    df["corr_eq"] = m["corr"]
    for c in range(3):
        df[f"snr_step5_db_c{c}"] = m["snr5"][:, c]
    return df

# ------------------------------------------------------------------ split and metadata
def split_indices(n):
    """Row indices of the 60/20/20 split of step3 (seeds 13 and 20), same call pattern as step3."""
    idx = np.arange(n)
    tr, rest = train_test_split(idx, train_size=0.6, random_state=13)
    va, te = train_test_split(rest, test_size=0.5, random_state=20)
    return tr, va, te

def check_split_equivalence(n=5000):
    """The index split must equal what step3 does to the arrays (X_train, Y_train)."""
    X = np.repeat(np.arange(n, dtype=float)[:, None, None], 2, axis=2); Y = X + 0.5
    a, b, c, d = train_test_split(X, Y, train_size=0.6, random_state=13)
    xv, xt, yv, yt = train_test_split(b, d, test_size=0.5, random_state=20)
    tr, va, te = split_indices(n)
    return bool(np.array_equal(xv[:, 0, 0].astype(int), va) and np.array_equal(xt[:, 0, 0].astype(int), te)
                and np.array_equal(a[:, 0, 0].astype(int), tr))

def _dec(arr):
    return np.array([x.decode() if isinstance(x, (bytes, np.bytes_)) else str(x) for x in arr], dtype=object)

def build_row_metadata(train_path, poha_path, stead_path):
    """snr, shift, quake_energy_frac and earthquake name for every row of the final file, via source_index."""
    cols = ["snr", "shift", "quake_energy_frac"]
    with h5py.File(train_path, "r") as f:
        src = f["source_index"][:]; n_poha = int(f.attrs["n_poha"]); n_stead = int(f.attrs["n_stead"])
    out = {c: np.full(len(src), np.nan) for c in cols}; names = np.empty(len(src), dtype=object)
    isp = src < n_poha
    assert src.min() >= 0 and src.max() < n_poha + n_stead, "source_index out of range"
    for p, mask, shift in ((poha_path, isp, 0), (stead_path, ~isp, n_poha)):
        with h5py.File(p, "r") as f:
            for c in cols + ["earthquake_name"]:
                assert c in f, f"{c} missing in {Path(p).name}; keys: {list(f.keys())}"
            idx = src[mask] - shift
            for c in cols:
                out[c][mask] = f[c][:][idx]
            names[mask] = _dec(f["earthquake_name"][:][idx])
    meta = pd.DataFrame(dict(file_row=np.arange(len(src)), source_index=src, source=np.where(isp, "POHA", "STEAD"), **out))
    meta["earthquake_name"] = names
    return meta

def load_rows(train_path, rows, chunk=10000):
    """Read arbitrary rows (in the order given) of X_train / Y_train by one sequential pass over the file."""
    rows = np.asarray(rows); order = np.argsort(rows); sr = rows[order]
    X = np.empty((len(rows), 600, 3)); Y = np.empty_like(X)
    with h5py.File(train_path, "r") as f:
        n = f["X_train"].shape[0]
        for lo in range(0, n, chunk):
            hi = min(lo + chunk, n); a, b = np.searchsorted(sr, lo), np.searchsorted(sr, hi)
            if b > a:
                sel = sr[a:b] - lo
                X[order[a:b]] = f["X_train"][lo:hi][sel]; Y[order[a:b]] = f["Y_train"][lo:hi][sel]
    return X, Y

# ------------------------------------------------------------------ statistics
def boot_ci_mean(x, n_boot=1000, rng=None, alpha=0.05, chunk=100):
    x = np.asarray(x, float); x = x[np.isfinite(x)]
    if len(x) == 0:
        return np.nan, np.nan, np.nan
    rng = rng or np.random.default_rng(BOOT_SEED)
    means = np.concatenate([x[rng.integers(0, len(x), (min(chunk, n_boot - i), len(x)))].mean(1) for i in range(0, n_boot, chunk)])
    return float(x.mean()), float(np.quantile(means, alpha / 2)), float(np.quantile(means, 1 - alpha / 2))

def boot_ci_median(x, n_boot=1000, rng=None, alpha=0.05, chunk=100):
    x = np.asarray(x, float); x = x[np.isfinite(x)]
    rng = rng or np.random.default_rng(BOOT_SEED)
    med = np.concatenate([np.median(x[rng.integers(0, len(x), (min(chunk, n_boot - i), len(x)))], axis=1) for i in range(0, n_boot, chunk)])
    return float(np.median(x)), float(np.quantile(med, alpha / 2)), float(np.quantile(med, 1 - alpha / 2))

def snr_edges(lo=0.1, hi=100.0, per_decade=4):
    return np.logspace(np.log10(lo), np.log10(hi), int(round(np.log10(hi / lo) * per_decade)) + 1)

def bin_index(snr, edges):
    b = np.searchsorted(edges, snr, side="right") - 1
    b[snr == edges[-1]] = len(edges) - 2
    b[(snr < edges[0]) | (snr > edges[-1]) | ~np.isfinite(snr)] = -1
    return b

def summarise_model(df):
    """Model-level numbers. Pooled medians follow step5 (every trace-component is one value)."""
    n5 = (len(df) // 256) * 256            # step5 drops the last incomplete batch of 256
    def pool(d, p): return d[[f"{p}_c0", f"{p}_c1", f"{p}_c2"]].to_numpy().ravel()
    e, nz = pool(df, "ev_eq"), pool(df, "ev_noise"); d5 = df.iloc[:n5]
    return dict(n_samples=len(df),
                median_ev_eq=float(np.median(e)), median_ev_noise=float(np.median(nz)),
                mean_ev_eq_clipped=float(np.clip(e, EV_CLIP, 1).mean()), mean_ev_noise_clipped=float(np.clip(nz, EV_CLIP, 1).mean()),
                frac_ev_eq_lt0=float((e < 0).mean()), frac_ev_noise_lt0=float((nz < 0).mean()),
                test_loss=float(df["mse_total"].mean()), mean_mse_eq=float(df["mse_eq"].mean()), mean_mse_noise=float(df["mse_noise"].mean()),
                mean_corr_eq=float(np.nanmean(df["corr_eq"])),
                median_ev_eq_step5subset=float(np.median(pool(d5, "ev_eq"))), median_ev_noise_step5subset=float(np.median(pool(d5, "ev_noise"))),
                test_loss_step5subset=float(d5["mse_total"].mean()))


## Part A: authors' pretrained models on the Hawaii example files (Phase 1.1)
Switch `RUN_PART_A`. Uses the preprocessing of `notebooks/apply_to_continuous_data.ipynb`. A model that fails to load prints its full error and the loop continues.

In [8]:
def preprocess_example_nb(path, highpass=0.5):
    """Same steps as notebooks/apply_to_continuous_data.ipynb (decimate to 10 Hz, optional highpass, global normalisation)."""
    import obspy
    tr = obspy.read(str(path)); tr.merge(fill_value=0)
    tr.decimate(int(round(tr[0].stats.sampling_rate / 10)))
    tr.filter("highpass", freq=highpass)
    npts = tr[0].stats.npts // 600 * 600
    wf = np.zeros((npts, 3))
    for i in range(3):
        wf[:, i] = tr[i].data[:npts]
    mean, std = wf.mean(0), wf.std(0)
    wn = np.reshape(((wf - mean) / (std + 1e-12))[:, np.newaxis, :], (-1, 600, 3))
    return tr, wf, wn, mean, std

def separate_windows(model, wn, mean, std, device, bs=256):
    """wn: (n_windows, 600, 3) normalised. Returns earthquake and noise waveforms in original units, (n*600, 3)."""
    Xt = np.ascontiguousarray(np.moveaxis(wn, 1, -1))
    pe, pn = predict(model, Xt, device, bs)
    eq = np.moveaxis(pe, 1, -1) * std + mean
    nz = np.moveaxis(pn, 1, -1) * std + mean
    return eq.reshape(-1, 3), nz.reshape(-1, 3)

def window_figure(raw, eq, nz, name, title):
    import matplotlib.pyplot as plt
    e = (eq.reshape(-1, 600, 3) ** 2).sum((1, 2)); k = int(np.argmax(e))
    sl = slice(k * 600, (k + 1) * 600); t = np.arange(600) / 10
    fig, ax = plt.subplots(3, 3, sharex=True, figsize=(10, 5))
    for c, lab in enumerate("ENZ"):
        sc = np.max(np.abs(raw[sl, c])) + 1e-12
        for j, (a, col, tt) in enumerate(((raw, "k", "Original"), (eq, "r", "Earthquake"), (nz, "b", "Noise"))):
            ax[c, j].plot(t, a[sl, c] / sc, color=col, lw=0.8)
            if c == 0: ax[c, j].set_title(tt)
        ax[c, 0].set_ylabel(lab)
    for j in range(3): ax[2, j].set_xlabel("Time in window (s)")
    fig.suptitle(f"{title} (window {k}, largest earthquake-branch energy)"); save_fig(fig, name); plt.close(fig)

if RUN_PART_A:
    import obspy, matplotlib; matplotlib.use("Agg")
    torch, mods = prepare_torch(); dev = get_device()
    data_dir = CODE_DIR/"notebooks"/"data_and_model"
    a = hashlib.sha256((data_dir/"Branch_Encoder_Decoder_LSTM_Model.pth").read_bytes()).hexdigest()
    b = hashlib.sha256((CODE_DIR/"trained_models"/"Branch_Encoder_Decoder_LSTM"/"Branch_Encoder_Decoder_LSTM_Model.pth").read_bytes()).hexdigest()
    print("LSTM file in notebooks/data_and_model identical to trained_models/:", a == b)
    events = obspy.read_events(str(data_dir/"catalog_local.20210731-20210901.xml"))
    status = {}
    for fname in ("HV.HAT.20210807-20210808.mseed", "IU.POHA.20210807-20210808.mseed"):
        tr, wf, wn, mean, std = preprocess_example_nb(data_dir/fname)
        print(fname, "channels:", [t.id for t in tr], "windows:", wn.shape[0])
        for bn in BOTTLENECKS:
            p = CODE_DIR/"trained_models"/f"Branch_Encoder_Decoder_{bn}"/f"Branch_Encoder_Decoder_{bn}_Model.pth"
            try:
                m = load_model(p, dev); ok, err = check_forward(m, dev)
                if not ok: raise RuntimeError(err)
                eq, nz = separate_windows(m, wn, mean, std, dev)
                window_figure(wf[:len(eq)], eq, nz, f"partA_{fname[:6]}_{bn}", f"{fname[:6]} pretrained {bn}")
                if bn == "LSTM":
                    for lab, arr in (("raw", wf[:len(eq)]), ("earthquake", eq), ("noise", nz)):
                        t2 = tr.copy()
                        t2[0].data = arr[:, 0]
                        try:
                            t2[0].plot(type="dayplot", interval=60, size=(800, 1200), vertical_scaling_range=5e3, events=events,
                                       outfile=str(FIGS/f"partA_dayplot_{fname[:6]}_{lab}.png"))
                        except Exception as e:
                            print("dayplot failed:", repr(e))
                status[f"{fname[:6]}_{bn}"] = "ok"
            except Exception:
                status[f"{fname[:6]}_{bn}"] = "FAILED"; print("FAILED", fname, bn); print(traceback.format_exc())
    json.dump(dict(status=status, compat_shim=COMPAT_LOG), open(RESULTS/"partA_status.json", "w"), indent=2)
    print(status, "| compat shim used for:", COMPAT_LOG)
    log_entry("A", "pretrained models on Hawaii files", f"{sum(v == 'ok' for v in status.values())}/{len(status)} ok")
else:
    print("Part A skipped")


Part A skipped


## Part B: training, one bottleneck and one seed per session
Switches `RUN_PART_B_SMOKE`, then `RUN_PART_B_FULL`. Set `BOTTLENECK` and `SEED` in the configuration cell.

- The smoke run trains `None` for 2 epochs on 5000 samples and projects the time of 100 epochs. The full run refuses to start if the projection exceeds `SAFETY_FACTOR * SESSION_LIMIT_H`.
- **No resume/checkpoint mechanism exists.** `step3` saves only at the end. A session that dies before epoch 100 loses the run. Adding resume would change the patched script, so it is not done without your decision.
- Commit mode (Save Version, Save and Run All) keeps running without the browser open, as I understand Kaggle; confirm on the Kaggle site.

In [9]:
SMOKE = {}
def run_step3(bottleneck, seed, epochs, subset=None, tag=None, log_name="train.log"):
    cmd = [sys.executable, "-u", "step3_training_1D.py", "--bottleneck", bottleneck, "--data", str(TRAIN_PATH),
           "--seed", str(seed), "--epochs", str(epochs)]
    if subset: cmd += ["--subset", str(subset)]
    if tag is not None: cmd += ["--tag", tag]
    print(" ".join(cmd)); t0 = time.time(); lines = []
    proc = subprocess.Popen(cmd, cwd=CODE_DIR, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    with open(LOGS/log_name, "w") as lf:
        for line in proc.stdout:
            print(line, end=""); lf.write(line); lines.append(line)
    return proc.wait(), "".join(lines), time.time() - t0

def gpu_warning():
    try:
        import torch
        if not torch.cuda.is_available():
            print("WARNING: no GPU visible. Turn on a GPU accelerator in the notebook settings."); return
        n = torch.cuda.get_device_name(0); print("GPU:", n)
        if "T4" in n:
            print("NOTE: T4 has very slow float64 (this model is float64). Check the smoke projection before a full run; a P100 is faster in float64.")
    except Exception:
        pass

if RUN_PART_B_SMOKE:
    assert TRAIN_PATH is not None, "attach the training dataset"
    gpu_warning()
    rc, out, dt = run_step3("None", 99, 2, subset=5000, tag="_smoke", log_name="smoke.log")
    assert rc == 0, "smoke run failed, read the traceback above"
    times = [float(x) for x in re.findall(r"time per epoch: ([\d.]+) s", out)]
    info = json.load(open(MODEL_DIR/"Branch_Encoder_Decoder_None_smoke"/"run_info.json"))
    with h5py.File(TRAIN_PATH, "r") as f: n_all = f["X_train"].shape[0]
    full_rows = int(round(n_all * 0.6)) + int(round(n_all * 0.2))      # train + validation rows of the full run (test rows are not used in training)
    scale = full_rows / (info["n_train"] + info["n_val"])
    t_ep = times[-1]                                                    # last epoch (epoch 1 includes CUDA start-up)
    proj_h = t_ep * scale * EPOCHS / 3600
    SMOKE.update(epoch_times=times, projected_hours=proj_h, ok=proj_h <= SAFETY_FACTOR * SESSION_LIMIT_H)
    print(f"\nsmoke epoch times (s): {times}; scale to full data x{scale:.1f}")
    print(f"PROJECTED time for {EPOCHS} epochs of the 'None' bottleneck: {proj_h:.1f} h (session limit {SESSION_LIMIT_H} h, safety {SAFETY_FACTOR}).")
    print("Projection is for 'None' (smallest model). LSTM/attention/Transformer are slower: scale up using their own smoke run if in doubt.")
    print("OK to start the full run" if SMOKE["ok"] else "STOP: projected time exceeds the session limit. Do not start the full run on this GPU.")
    json.dump(SMOKE, open(RESULTS/"smoke_projection.json", "w"), indent=2)
    log_entry("B smoke", "step3 --bottleneck None --epochs 2 --subset 5000 --tag _smoke", f"epoch times {times}, projected {proj_h:.1f} h", 99)
else:
    print("smoke run skipped")

if RUN_PART_B_FULL:
    assert TRAIN_PATH is not None, "attach the training dataset"
    if not SMOKE.get("ok") and not FORCE_FULL_RUN:
        raise RuntimeError("Run the smoke test first in this session (RUN_PART_B_SMOKE) and read the projection, or set FORCE_FULL_RUN = True.")
    gpu_warning()
    rc, out, dt = run_step3(BOTTLENECK, SEED, EPOCHS, log_name=f"train_{BOTTLENECK}_seed{SEED}.log")
    assert rc == 0, "training failed, read the traceback above"
    folder = run_folder_name(BOTTLENECK, SEED); src_dir = MODEL_DIR/folder
    info = json.load(open(src_dir/"run_info.json"))
    shutil.copytree(src_dir, RUNS_OUT/folder, dirs_exist_ok=True)
    write_run_config(dict(dir=RUNS_OUT/folder, info=info))
    size_mb = sum(p.stat().st_size for p in (RUNS_OUT/folder).rglob("*")) / 1e6
    print(f"\nDONE {folder}: {info['runtime_s']/60:.1f} min, {info['n_params']} params, final val loss {info['final_val_loss']:.5f}")
    print(f"copied to {RUNS_OUT/folder} ({size_mb:.1f} MB). Save a notebook version, then make a dataset from the output (see the top of the notebook).")
    print(update_run_table().to_string())
    log_entry("B full", f"step3 --bottleneck {BOTTLENECK} --seed {SEED} --epochs {EPOCHS}", f"{info['runtime_s']/60:.1f} min, val loss {info['final_val_loss']:.5f}", SEED)
else:
    print("full training skipped")


GPU: Tesla T4
NOTE: T4 has very slow float64 (this model is float64). Check the smoke projection before a full run; a P100 is faster in float64.
/usr/bin/python3 -u step3_training_1D.py --bottleneck None --data /kaggle/input/datasets/prabhanshu22/wdn-training-data/training_datasets_all_snr_40_unshuffled.hdf5 --seed 99 --epochs 2 --subset 5000 --tag _smoke
device: cuda:0 Tesla T4
############ Loading data ############
data: (5000, 600, 3) float64
Branch_Encoder_Decoder_None: 78090 parameters
############ training model Branch_Encoder_Decoder_None ############
[1/2] train_loss: 2.95172 valid_loss: 1.31331
time per epoch: 7.111 s
[2/2] train_loss: 2.70934 valid_loss: 2.11543
time per epoch: 2.105 s
Training is done! 0.2 min
saved to /kaggle/working/Model_and_datasets_1D_all_snr_40_unshuffled_equal_epoch100/Branch_Encoder_Decoder_None_smoke

smoke epoch times (s): [7.111, 2.105]; scale to full data x37.4
PROJECTED time for 100 epochs of the 'None' bottleneck: 2.2 h (session limit 12.0 h, s

## Part C: evaluation on the shared test split (Phase 1.4)
Switch `RUN_PART_C`. Attach the data dataset (final file **and** both intermediate files) and every run dataset. All models and the two baselines are scored on the same 37,400 test samples, in the order given by the step3 split. Per-sample results go to `results/per_sample_<model>.csv.gz`; figures are drawn from the CSV files.

In [10]:
REG_PATH = RESULTS/"model_registry.csv"

def load_registry():
    if not REG_PATH.exists():
        return {}
    df = read_csv_keep(REG_PATH); df["bottleneck"] = df["bottleneck"].fillna("")
    return df.set_index("model_id").to_dict("index")

def save_registry(reg):
    pd.DataFrame([dict(model_id=k, **v) for k, v in reg.items()]).to_csv(REG_PATH, index=False)

def discover_models():
    models = []
    for (b, s), r in sorted(discover_runs(EPOCHS_REQUIRED).items()):
        i = r["info"]
        models.append(dict(id=f"ours_{b}_seed{s}", kind="ours", bottleneck=b, seed=s, path=r["model_file"], n_params=i.get("n_params"),
                           runtime_s=i.get("runtime_s"), gpu=i.get("gpu"), torch=i.get("torch"),
                           final_train_loss=i.get("final_train_loss"), final_val_loss=i.get("final_val_loss"), hist_dir=r["dir"]))
    for b in BOTTLENECKS:
        d = CODE_DIR/"trained_models"/f"Branch_Encoder_Decoder_{b}"
        models.append(dict(id=f"pretrained_{b}", kind="pretrained", bottleneck=b, seed=-1,
                           path=d/f"Branch_Encoder_Decoder_{b}_Model.pth", hist_dir=d))
    return models

if RUN_PART_C:
    assert TRAIN_PATH is not None and POHA_PATH is not None and STEAD_PATH is not None, \
        "Part C needs the final file AND both intermediate mixture files (per-sample SNR/shift metadata)."
    assert check_split_equivalence(), "index split differs from the step3 array split"
    N_ALL = DATA_FACTS["n_rows"]
    tr_idx, va_idx, te_idx = split_indices(N_ALL)
    print(f"split: train {len(tr_idx)}, validation {len(va_idx)}, test {len(te_idx)}; index split == step3 array split: True")
    np.savez_compressed(RESULTS/"split_indices.npz", train=tr_idx, val=va_idx, test=te_idx)

    ROW_META = build_row_metadata(TRAIN_PATH, POHA_PATH, STEAD_PATH)
    meta = ROW_META.iloc[te_idx].reset_index(drop=True); meta.insert(0, "row", np.arange(len(meta)))
    meta.to_csv(RESULTS/"test_metadata.csv.gz", index=False)
    json.dump(dict(n_test=len(meta), frac_energy_lt_1pct=float((meta.quake_energy_frac < 0.01).mean()),
                   frac_poha=float((meta.source == "POHA").mean()),
                   snr_range=[float(meta.snr.min()), float(meta.snr.max())]), open(RESULTS/"test_set_facts.json", "w"), indent=2)
    t0 = time.time(); Xte, Yte = load_rows(TRAIN_PATH, te_idx)
    Xt = np.ascontiguousarray(np.moveaxis(Xte, 1, -1)); Yt = np.ascontiguousarray(np.moveaxis(Yte, 1, -1)); del Xte, Yte
    print("test data loaded:", Xt.shape, f"{time.time() - t0:.0f} s")

    torch, mods = prepare_torch(); dev = get_device(); REG = load_registry(); errors = []
    def finish(mid, pe, pn, info):
        df = per_sample_frame(meta, metrics_chunked(Xt, Yt, pe, pn))
        df.to_csv(RESULTS/f"per_sample_{mid}.csv.gz", index=False)
        REG[mid] = {**REG.get(mid, {}), **info, "status": "ok"}; save_registry(REG)

    for m in discover_models():
        out = RESULTS/f"per_sample_{m['id']}.csv.gz"
        if out.exists() and not C_RECOMPUTE:
            print("reuse", m["id"]); continue
        try:
            model = load_model(m["path"], dev); ok, err = check_forward(model, dev)
            if not ok: raise RuntimeError(err)
            pe, pn = predict(model, Xt, dev)
            npar = int(sum(p.numel() for p in model.parameters()))
            info = {k: m.get(k) for k in ("kind", "bottleneck", "seed", "runtime_s", "gpu", "torch", "final_train_loss", "final_val_loss")}
            finish(m["id"], pe, pn, {**info, "n_params": npar, "hist_dir": str(m["hist_dir"]), "model_path": str(m["path"])})
            print("evaluated", m["id"], f"({npar} params)")
            del model, pe, pn
        except Exception:
            errors.append(m["id"]); REG[m["id"]] = dict(kind=m["kind"], bottleneck=m["bottleneck"], seed=m["seed"], status="FAILED")
            save_registry(REG); print("FAILED", m["id"]); print(traceback.format_exc())
            with open(LOGS/"load_errors.txt", "a") as f: f.write(f"{m['id']}\n{traceback.format_exc()}\n")
    # baselines (same metrics, same test set)
    from scipy.signal import butter, sosfiltfilt
    sos = butter(4, list(BANDPASS_HZ), btype="band", fs=10, output="sos")
    bp = sosfiltfilt(sos, Xt, axis=-1)
    for mid, pe, pn, desc in (("baseline_identity", Xt, Xt, "earthquake output = input, noise output = input"),
                               ("baseline_bandpass", bp, Xt - bp, f"zero-phase 4th-order Butterworth band-pass {BANDPASS_HZ} Hz; noise = input - filtered")):
        if (RESULTS/f"per_sample_{mid}.csv.gz").exists() and not C_RECOMPUTE: continue
        finish(mid, pe, pn, dict(kind="baseline", bottleneck=mid.replace("baseline_", ""), seed=-1, n_params=0, note=desc))
        print("baseline", mid)
    del Xt, Yt
    print("failed models:", errors, "| compat shim used for:", COMPAT_LOG)
    json.dump(dict(failed=errors, compat_shim=COMPAT_LOG), open(RESULTS/"partC_eval_status.json", "w"), indent=2)
    log_entry("C eval", "evaluate runs + pretrained + baselines on the shared test split", f"failed={errors}, shim={COMPAT_LOG}")
else:
    print("Part C skipped")


Part C skipped


In [11]:
STATS = ["median_ev_eq", "median_ev_noise", "mean_ev_eq_clipped", "mean_ev_noise_clipped", "frac_ev_eq_lt0", "frac_ev_noise_lt0",
         "test_loss", "mean_corr_eq", "median_ev_eq_step5subset", "test_loss_step5subset"]
PAPER_KEY = {"median_ev_eq": "ev_earthquake", "median_ev_noise": "ev_noise", "test_loss": "test_loss"}

def load_ps(mid):
    return read_csv_keep(RESULTS/f"per_sample_{mid}.csv.gz")

if RUN_PART_C:
    REG = load_registry(); ids = [k for k, v in REG.items() if v.get("status") == "ok" and (RESULTS/f"per_sample_{k}.csv.gz").exists()]
    rows = []
    for mid in ids:
        r = REG[mid]; rows.append(dict(model_id=mid, kind=r["kind"], bottleneck=r.get("bottleneck", ""), seed=r.get("seed", -1),
                                       n_params=r.get("n_params"), **summarise_model(load_ps(mid))))
    SUMMARY = pd.DataFrame(rows); SUMMARY.to_csv(RESULTS/"table_ev_per_seed.csv", index=False)

    agg = []
    for (kind, b), g in SUMMARY.groupby(["kind", "bottleneck"]):
        d = dict(kind=kind, bottleneck=b, n_runs=len(g), n_params=g.n_params.iloc[0])
        for s in STATS:
            d[f"{s}_mean"] = g[s].mean(); d[f"{s}_std"] = g[s].std(ddof=1) if len(g) > 1 else np.nan
        if kind == "ours":
            for s, k in PAPER_KEY.items(): d[f"paper_{s}"] = PAPER_VALUES[k].get(b)
            d["paper_n_params"] = PAPER_VALUES["n_params"].get(b)
        agg.append(d)
    AGG = pd.DataFrame(agg); AGG.to_csv(RESULTS/"table_ev_per_bottleneck.csv", index=False)
    print("Median EV of the earthquake branch (pooled trace-components, as in step5), mean +/- std over seeds:")
    print(AGG[["kind", "bottleneck", "n_runs", "median_ev_eq_mean", "median_ev_eq_std", "median_ev_noise_mean", "test_loss_mean", "n_params"]].round(4).to_string())

    # ours vs the authors' pretrained models on the same test set
    o = AGG[AGG.kind == "ours"].set_index("bottleneck"); pr = AGG[AGG.kind == "pretrained"].set_index("bottleneck"); cmp_rows = []
    for b in BOTTLENECKS:
        if b in o.index and b in pr.index:
            cmp_rows.append(dict(bottleneck=b, **{f"ours_{s}": o.loc[b, f"{s}_mean"] for s in ("median_ev_eq", "median_ev_noise", "test_loss")},
                                 **{f"pretrained_{s}": pr.loc[b, f"{s}_mean"] for s in ("median_ev_eq", "median_ev_noise", "test_loss")},
                                 note="pretrained models were trained on a split we cannot see: an external reference, not a controlled comparison"))
    pd.DataFrame(cmp_rows).to_csv(RESULTS/"table_ours_vs_pretrained.csv", index=False)

    # parameters, runtime
    pd.DataFrame([dict(model_id=k, **{c: v.get(c) for c in ("kind", "bottleneck", "seed", "n_params", "runtime_s", "gpu", "torch", "final_train_loss", "final_val_loss")})
                  for k, v in REG.items()]).to_csv(RESULTS/"table_params_runtime.csv", index=False)

    # what the authors stored in the repo (their own test split, their step5): reference only
    ap = CODE_DIR/"trained_models"/"all_model_comparison"/"all_model_comparison.hdf5"
    if ap.exists():
        with h5py.File(ap, "r") as f:
            names = [x.decode() for x in f["model_names"][:]]; a_eq = f["model_mse_earthquake_all"][:]; a_nz = f["model_mse_noise_all"][:]
            a_loss = f["model_test_loss_all"][:]; a_par = f["model_param_number"][:]
        pd.DataFrame(dict(model=names, median_ev_eq=np.median(a_eq, 1), median_ev_noise=np.median(a_nz, 1),
                          raw_mean_ev_eq=a_eq.mean(1), frac_ev_eq_lt0=(a_eq < 0).mean(1), test_loss=a_loss, n_params=a_par,
                          note="from the repo file trained_models/all_model_comparison, authors' own test split; raw means are dominated by extreme negative EV values")
                     ).to_csv(RESULTS/"authors_stored_reference.csv", index=False)
        print("authors' stored reference written (raw mean EV is dominated by extreme outliers; use medians)")
else:
    print("Part C tables skipped")


Part C tables skipped


In [12]:
def ev_vs_snr_tables(curves, edges, n_boot):
    """curves: dict name -> DataFrame (per-sample values, same test rows). Returns one long table."""
    meta = pd.read_csv(RESULTS/"test_metadata.csv.gz"); b = bin_index(meta.snr.to_numpy(), edges)
    groups = {"all": np.ones(len(meta), bool), "POHA": (meta.source == "POHA").to_numpy(), "STEAD": (meta.source == "STEAD").to_numpy(),
              "energy_lt1pct": (meta.quake_energy_frac < 0.01).to_numpy(), "energy_ge1pct": (meta.quake_energy_frac >= 0.01).to_numpy()}
    rng = np.random.default_rng(BOOT_SEED); rows = []
    for name, df in curves.items():
        for kind, col in (("eq", "ev_eq_mean_clipped"), ("noise", "ev_noise_mean_clipped")):
            v = df[col].to_numpy()
            for gname, gm in groups.items():
                for k in range(len(edges) - 1):
                    x = v[gm & (b == k)]
                    mean, lo, hi = boot_ci_mean(x, n_boot, rng) if len(x) > 1 else (np.nan, np.nan, np.nan)
                    rows.append(dict(curve=name, ev_kind=kind, group=gname, bin_lo=edges[k], bin_hi=edges[k + 1], n=len(x),
                                     mean_ev_clipped=mean, ci_lo=lo, ci_hi=hi, median_ev=float(np.median(x)) if len(x) else np.nan))
    return pd.DataFrame(rows)

def step5_style_table(curves):
    """Median EV in bins of log10(SNR) of width 0.5 between -2 and 4, per trace-component, as in step5 (pooled over seeds)."""
    edges = np.arange(-2, 4, 0.5); rows = []
    for name, dfs in curves.items():
        for kind, p in (("eq", "ev_eq"), ("noise", "ev_noise")):
            ev = np.concatenate([d[[f"{p}_c{c}" for c in range(3)]].to_numpy().ravel() for d in dfs])
            sn = np.concatenate([d[[f"snr_step5_db_c{c}" for c in range(3)]].to_numpy().ravel() / 10 for d in dfs])
            for e in edges:
                x = ev[(sn >= e) & (sn <= e + 0.5)]
                if len(x) == 0: continue
                med = np.median(x); up = x[x > med]; dn = x[x < med]
                rows.append(dict(curve=name, ev_kind=kind, bin_center=e + 0.25, n=len(x), median_ev=med,
                                 err_up=(np.median(up) - med) if len(up) else 0.0, err_down=(med - np.median(dn)) if len(dn) else 0.0))
    return pd.DataFrame(rows)

if RUN_PART_C:
    REG = load_registry(); ok = [k for k, v in REG.items() if v.get("status") == "ok"]
    # curves: seed-mean per-sample EV for ours (per bottleneck), pretrained, baselines
    curves, step5_curves = {}, {}
    for b in BOTTLENECKS:
        mine = [k for k in ok if k.startswith(f"ours_{b}_seed")]
        if mine:
            dfs = [load_ps(k) for k in mine]; base = dfs[0].copy()
            for c in ("ev_eq_mean_clipped", "ev_noise_mean_clipped"):
                base[c] = np.mean([d[c].to_numpy() for d in dfs], axis=0)
            curves[f"ours_{b}_seedmean"] = base; step5_curves[f"ours_{b}"] = dfs
        if f"pretrained_{b}" in ok:
            d = load_ps(f"pretrained_{b}"); curves[f"pretrained_{b}"] = d; step5_curves[f"pretrained_{b}"] = [d]
    for k in ("baseline_identity", "baseline_bandpass"):
        if k in ok: curves[k] = load_ps(k)
    CURVES = curves
    edges = snr_edges(0.1, 100, SNR_BINS_PER_DECADE)
    t0 = time.time(); ev_vs_snr_tables(curves, edges, N_BOOT).to_csv(RESULTS/"table_ev_vs_snr.csv", index=False)
    step5_style_table(step5_curves).to_csv(RESULTS/"table_ev_vs_log_snr_step5style.csv", index=False)
    print(f"EV vs SNR tables written ({time.time() - t0:.0f} s)")

    # paired comparisons between bottlenecks (seed-mean per-sample EV, same test samples), bootstrap over test samples
    rng = np.random.default_rng(BOOT_SEED); names = [n for n in curves if n.endswith("_seedmean")]; rows = []
    for i in range(len(names)):
        for j in range(i + 1, len(names)):
            a, b_ = curves[names[i]]["ev_eq_mean_clipped"].to_numpy(), curves[names[j]]["ev_eq_mean_clipped"].to_numpy(); d = a - b_
            mean, lo, hi = boot_ci_mean(d, N_BOOT, rng); med, mlo, mhi = boot_ci_median(d, N_BOOT, rng)
            rows.append(dict(A=names[i], B=names[j], n=len(d), mean_diff=mean, mean_ci_lo=lo, mean_ci_hi=hi, median_diff=med,
                             median_ci_lo=mlo, median_ci_hi=mhi, frac_A_better=float((d > 0).mean()), frac_equal=float((d == 0).mean()),
                             note="CI covers test-sample sampling only; seed-to-seed spread is in table_ev_per_seed.csv"))
    pd.DataFrame(rows).to_csv(RESULTS/"table_paired_comparisons.csv", index=False)

    # leakage audit: do test samples share a source draw or an earthquake trace with training samples?
    tr_idx, va_idx, te_idx = split_indices(DATA_FACTS["n_rows"]); R = ROW_META
    def share(idx_a, idx_b, col):
        s = set(R[col].iloc[idx_b]); return float(np.mean([x in s for x in R[col].iloc[idx_a]]))
    leak = {}
    for nm, ia in (("test", te_idx), ("validation", va_idx)):
        for col in ("source_index", "earthquake_name"):
            leak[f"frac_{nm}_rows_sharing_{col}_with_train"] = share(ia, tr_idx, col)
    for src in ("POHA", "STEAD"):
        m = (R.source.iloc[te_idx] == src).to_numpy(); leak[f"frac_test_{src}_rows_sharing_earthquake_name_with_train"] = float(
            np.mean([x in set(R.earthquake_name.iloc[tr_idx]) for x in R.earthquake_name.iloc[te_idx][m]]))
    leak["note"] = "the repo splits after mixing, so the same source trace can sit in train and test with different noise and shift"
    json.dump(leak, open(RESULTS/"leakage_audit.json", "w"), indent=2); print(json.dumps(leak, indent=2))

    # failure cases and examples (reference model)
    ref = FAILURE_MODEL_ID or next((k for k in ok if k.startswith("ours_LSTM")), next((k for k in ok if k.startswith("ours_")), None))
    if ref:
        df = load_ps(ref); raw = df[[f"ev_eq_c{c}" for c in range(3)]].mean(1).to_numpy()
        order = np.lexsort((raw, df.ev_eq_mean_clipped.to_numpy())); nfail = max(1, int(FAIL_FRACTION * len(df)))
        fail = df.iloc[order[:nfail]].copy(); fail.to_csv(RESULTS/f"failure_cases_{ref}.csv", index=False)
        fs = dict(model=ref, n_failures=nfail, frac_energy_lt_1pct=float((fail.quake_energy_frac < 0.01).mean()),
                  frac_poha=float((fail.source == "POHA").mean()), median_snr=float(fail.snr.median()),
                  test_frac_energy_lt_1pct=float((df.quake_energy_frac < 0.01).mean()))
        json.dump(fs, open(RESULTS/"failure_cases_summary.json", "w"), indent=2); print(fs)
        # examples: good = random from the best 10 % with >=1 % energy; average = nearest to the median; failure = random from the worst set
        rng = np.random.default_rng(EXAMPLE_SEED); sig = df.quake_energy_frac.to_numpy() >= 0.01; v = df.ev_eq_mean_clipped.to_numpy()
        good_pool = np.where(sig & (v >= np.quantile(v[sig], 0.9)))[0]; med = np.median(v[sig])
        avg_pool = np.where(sig)[0][np.argsort(np.abs(v[sig] - med))[:200]]
        sel = {"good": rng.choice(good_pool, 3, replace=False), "average": rng.choice(avg_pool, 3, replace=False),
               "failure": rng.choice(order[:nfail], 3, replace=False)}
        idx_all = np.concatenate(list(sel.values())); te_rows = te_idx[idx_all]
        Xe, Ye = load_rows(TRAIN_PATH, te_rows); Xe = np.moveaxis(Xe, 1, -1); Ye = np.moveaxis(Ye, 1, -1)
        mdl = load_model(REG[ref]["model_path"], dev); check_forward(mdl, dev); pe, pn = predict(mdl, np.ascontiguousarray(Xe), dev)
        np.savez_compressed(RESULTS/f"example_waveforms_{ref}.npz", X=Xe, Y=Ye, pred_eq=pe, pred_noise=pn,
                            kind=np.repeat(list(sel.keys()), 3), test_row=idx_all, ev=v[idx_all], snr=df.snr.to_numpy()[idx_all])
    log_entry("C analysis", "tables, bootstrap CIs, leakage audit, failure cases", f"reference model {ref}")
else:
    print("Part C analysis skipped")


Part C analysis skipped


In [13]:
def fig_ev_bars(agg):
    import matplotlib.pyplot as plt
    o = agg[agg.kind == "ours"].set_index("bottleneck"); pr = agg[agg.kind == "pretrained"].set_index("bottleneck")
    fig, ax = plt.subplots(1, 2, figsize=(11, 4))
    for a, s, ttl in ((ax[0], "median_ev_eq", "Earthquake branch"), (ax[1], "median_ev_noise", "Noise branch")):
        x = np.arange(len(BOTTLENECKS))
        a.bar(x - 0.2, [o[f"{s}_mean"].get(b, np.nan) for b in BOTTLENECKS], 0.4, yerr=[np.nan_to_num(o[f"{s}_std"].get(b, np.nan)) for b in BOTTLENECKS], label="ours (mean, std over seeds)")
        a.bar(x + 0.2, [pr[f"{s}_mean"].get(b, np.nan) for b in BOTTLENECKS], 0.4, label="authors' pretrained")
        pv = PAPER_VALUES["ev_earthquake" if "eq" in s else "ev_noise"]
        a.plot(x, [np.nan if pv[b] is None else pv[b] for b in BOTTLENECKS], "k*", ms=10, label="paper (if entered)")
        a.set_xticks(x); a.set_xticklabels(BOTTLENECKS); a.set_ylabel("median EV"); a.set_title(ttl); a.grid(alpha=.3)
    ax[0].legend(fontsize=8); return fig

def fig_ev_vs_snr(tab, group, kind):
    import matplotlib.pyplot as plt
    t = tab[(tab.group == group) & (tab.ev_kind == kind)]; fig, ax = plt.subplots(figsize=(7, 4.5)); cols = plt.cm.tab10.colors
    for i, b in enumerate(BOTTLENECKS):
        for name, ls in ((f"ours_{b}_seedmean", "-"), (f"pretrained_{b}", ":")):
            d = t[t.curve == name]
            if len(d):
                xc = np.sqrt(d.bin_lo * d.bin_hi); ax.plot(xc, d.mean_ev_clipped, ls, color=cols[i], marker="s" if ls == "-" else None, ms=3, label=name)
                if ls == "-": ax.fill_between(xc, d.ci_lo, d.ci_hi, color=cols[i], alpha=.15)
    for name, col in (("baseline_identity", "gray"), ("baseline_bandpass", "k")):
        d = t[t.curve == name]
        if len(d): ax.plot(np.sqrt(d.bin_lo * d.bin_hi), d.mean_ev_clipped, "--", color=col, label=name)
    ax.set_xscale("log"); ax.set_xlabel("input SNR (linear amplitude factor of the mixture)"); ax.set_ylabel(f"mean EV ({kind}, clipped at {EV_CLIP})")
    ax.set_title(f"group: {group}"); ax.grid(alpha=.3); ax.legend(fontsize=6, ncol=2); return fig

def fig_step5_style(tab):
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(1, 2, figsize=(12, 4.5), sharey=True); cols = plt.cm.tab10.colors
    for a, kind in zip(ax, ("eq", "noise")):
        for i, b in enumerate(BOTTLENECKS):
            d = tab[(tab.curve == f"ours_{b}") & (tab.ev_kind == kind)]
            if len(d): a.errorbar(d.bin_center + i * 0.03, d.median_ev, yerr=[d.err_down, d.err_up], marker="s", ms=3, lw=1, color=cols[i], label=b)
        a.set_ylim(-0.4, 1.1); a.set_xlabel("log10(SNR) (power ratio of the realised window)"); a.set_title(kind); a.grid(alpha=.3)
    ax[0].set_ylabel("median EV"); ax[0].legend(); return fig

def fig_curves():
    import matplotlib.pyplot as plt
    cols = plt.cm.tab10.colors; fig, ax = plt.subplots(3, len(BOTTLENECKS), figsize=(3.2 * len(BOTTLENECKS), 8), sharey="row", squeeze=False)
    REG = load_registry()
    for j, b in enumerate(BOTTLENECKS):
        for mid, r in REG.items():
            if r.get("bottleneck") != b or r.get("kind") not in ("ours", "pretrained") or not isinstance(r.get("hist_dir"), str): continue
            hp = Path(r["hist_dir"])/f"Branch_Encoder_Decoder_{b}_Training_history.hdf5"
            if not hp.exists(): continue
            with h5py.File(hp, "r") as f:
                for i, (tr_k, va_k) in enumerate((("loss", "val_loss"), ("earthquake_loss", "earthquake_val_loss"), ("noise_loss", "noise_val_loss"))):
                    c = "gray" if r["kind"] == "pretrained" else cols[int(r["seed"]) % 10]
                    ax[i, j].semilogy(f[tr_k][:], "--", color=c, lw=.8); ax[i, j].semilogy(f[va_k][:], "-", color=c, lw=1.2, label=mid.replace("ours_", ""))
        ax[0, j].set_title(b); ax[2, j].set_xlabel("epoch")
    for i, l in enumerate(("total MSE", "earthquake branch MSE", "noise branch MSE")): ax[i, 0].set_ylabel(l + "\n(dashed train, solid validation)")
    ax[0, -1].legend(fontsize=6); return fig

def fig_failure_grid(ref):
    import matplotlib.pyplot as plt
    z = np.load(RESULTS/f"example_waveforms_{ref}.npz", allow_pickle=True); n = len(z["kind"]); t = np.arange(600) / 10
    fig, ax = plt.subplots(n, 3, figsize=(13, 2.1 * n))
    for r in range(n):
        c = 2; sc = np.max(np.abs(z["X"][r, c])) + 1e-12
        ax[r, 0].plot(t, z["X"][r, c] / sc, "k", lw=.7, label="input"); ax[r, 0].plot(t, z["Y"][r, c] / sc, "r", lw=.7, label="true earthquake")
        ax[r, 1].plot(t, z["Y"][r, c] / sc, "r", lw=.7); ax[r, 1].plot(t, z["pred_eq"][r, c] / sc, "b", lw=.7, label="predicted")
        f = np.fft.rfftfreq(600, .1)[1:]
        for a, lab, col in ((z["Y"][r, c], "true eq", "r"), (z["pred_eq"][r, c], "pred eq", "b"), (z["X"][r, c] - z["Y"][r, c], "true noise", "gray"), (z["pred_noise"][r, c], "pred noise", "c")):
            ax[r, 2].loglog(f, np.abs(np.fft.rfft(a / sc))[1:], col, lw=.6, label=lab)
        ax[r, 0].set_ylabel(f"{z['kind'][r]}\nEV {z['ev'][r]:.2f}\nSNR {z['snr'][r]:.2g}", fontsize=7)
    ax[0, 0].legend(fontsize=6); ax[0, 1].legend(fontsize=6); ax[0, 2].legend(fontsize=6)
    ax[-1, 0].set_xlabel("time (s), component Z"); ax[-1, 2].set_xlabel("frequency (Hz)"); return fig

if RUN_PART_C:
    import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
    AGG = read_csv_keep(RESULTS/"table_ev_per_bottleneck.csv"); TAB = read_csv_keep(RESULTS/"table_ev_vs_snr.csv"); S5 = read_csv_keep(RESULTS/"table_ev_vs_log_snr_step5style.csv")
    for fig, name in ((fig_ev_bars(AGG), "fig_ev_per_bottleneck"), (fig_step5_style(S5), "fig_ev_vs_logsnr_step5style"), (fig_curves(), "fig_training_curves")):
        save_fig(fig, name); plt.close(fig)
    for g in ("all", "POHA", "STEAD", "energy_lt1pct", "energy_ge1pct"):
        for k in ("eq", "noise"):
            fig = fig_ev_vs_snr(TAB, g, k); save_fig(fig, f"fig_ev_vs_snr_{k}_{g}"); plt.close(fig)
    for p in RESULTS.glob("example_waveforms_*.npz"):
        fig = fig_failure_grid(p.stem.replace("example_waveforms_", "")); save_fig(fig, f"fig_examples_{p.stem[18:]}"); plt.close(fig)
    print("figures written to", FIGS); log_entry("C figures", "figures from results/*.csv", "ok")
else:
    print("Part C figures skipped")


Part C figures skipped


## Part D: real continuous data (step6 and step7a logic)
Switch `RUN_PART_D`. One-day Hawaii example file. Differences from the repo scripts: the one-day file is truncated to a multiple of 600 samples, catalogue origin times replace TauP P arrivals (no station inventory in the repo), and step7b is not run.

In [14]:
if RUN_PART_D:
    import obspy, matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
    from obspy import Stream
    from obspy.signal.trigger import recursive_sta_lta, coincidence_trigger
    torch, mods = prepare_torch(); dev = get_device(); U = mods["utilities"]
    outd = RESULTS/"real_data"; outd.mkdir(exist_ok=True)
    # ---- model
    if D_MODEL_SOURCE == "ours":
        run = discover_runs(EPOCHS_REQUIRED).get((D_BOTTLENECK, D_SEED)); assert run, f"no finished run for {D_BOTTLENECK}, seed {D_SEED} is attached"
        mpath, label = run["model_file"], f"ours_{D_BOTTLENECK}_seed{D_SEED}"
    else:
        mpath = CODE_DIR/"trained_models"/f"Branch_Encoder_Decoder_{D_BOTTLENECK}"/f"Branch_Encoder_Decoder_{D_BOTTLENECK}_Model.pth"; label = f"pretrained_{D_BOTTLENECK}"
    model = load_model(mpath, dev); ok, err = check_forward(model, dev); assert ok, err
    # ---- data (step6 logic on the one-day example file; truncated to a multiple of 600 samples, as the example notebook does)
    dpath = CODE_DIR/"notebooks"/"data_and_model"/D_STATION_FILE
    if D_PREPROCESS == "example_nb":
        tr, wf, wn, mean, std = preprocess_example_nb(dpath); f_s = 10.0
    else:
        tr = obspy.read(str(dpath)); tr.merge(fill_value=0)
        print("traces:", [t.id for t in tr], "fs:", tr[0].stats.sampling_rate)
        assert len(tr) == 3 and len({t.stats.npts for t in tr}) == 1, "expected 3 equally long traces"
        w0 = np.stack([tr[i].data for i in range(3)], 1).astype(float); t0_ = np.arange(len(w0)) * tr[0].stats.delta
        _, wf, _ = U.downsample_series(t0_, w0, 10); n6 = len(wf) // 600 * 600; wf = wf[:n6]
        mean, std = wf.mean(0), wf.std(0); wn = np.reshape(((wf - mean) / (std + 1e-12))[:, np.newaxis, :], (-1, 600, 3)); f_s = 10.0
    eq, nz = separate_windows(model, wn, mean, std, dev); raw = wf[:len(eq)]; resid = raw - eq - nz
    with h5py.File(outd/f"{D_STATION_FILE[:6]}_{label}_processed_waveforms.hdf5", "w") as f:
        for k, v in (("waveform_time", np.arange(len(raw)) / f_s), ("waveform_original", raw), ("waveform_recovered", eq), ("noise_recovered", nz)):
            f.create_dataset(k, data=v)
    # ---- catalogue origin times (no station inventory in the repo, so P arrivals via TauP are not computed)
    t_start = tr[0].stats.starttime; ev_t = []
    for ev in obspy.read_events(str(CODE_DIR/"notebooks"/"data_and_model"/"catalog_local.20210731-20210901.xml")):
        try:
            if ev.magnitudes[0].mag >= 2.0: ev_t.append((ev.origins[0].time - t_start, ev.magnitudes[0].mag))
        except Exception: pass
    ev_t = np.array([e for e in ev_t if 0 <= e[0] <= len(raw) / f_s]).reshape(-1, 2)
    hrs = np.arange(len(raw))[::10] / f_s / 3600
    for c, lab in enumerate("ENZ"):
        fig, ax = plt.subplots(4, 1, figsize=(14, 9), sharex=True)
        for a, arr, col, tt in zip(ax, (raw, eq, nz, resid), "krbg", ("raw", "separated earthquake", "separated noise", "residual")):
            a.plot(hrs, arr[::10, c], color=col, lw=.4); a.set_ylabel(tt)
            for t in ev_t[:, 0]: a.axvline(t / 3600, color="m", alpha=.3, lw=.6)
        ax[-1].set_xlabel("hours since start of file"); ax[0].set_title(f"{D_STATION_FILE[:6]} {label} component {lab} (magenta: catalogue M>=2 origin times)")
        save_fig(fig, f"partD_{D_STATION_FILE[:6]}_{label}_{lab}"); plt.close(fig)
    # ---- STA/LTA as in step7a
    P = D_STALTA
    def mk(arr):
        s = tr.copy()
        for i in range(3):
            s[i].data = (arr[:, i] / D_SCALE).astype("float32"); s[i].stats.sampling_rate = f_s
        return s
    st_raw, st_eq = mk(raw), mk(eq)
    trig = {}
    for nm, st in (("raw", st_raw), ("separated", st_eq)):
        trig[nm] = coincidence_trigger("recstalta", P["trigger_on"], P["trigger_off"], st, P["threshold_coincidence"], sta=P["short_term"], lta=P["long_term"])
        pd.DataFrame([dict(time=str(t["time"]), seconds=t["time"] - t_start, duration=t["duration"]) for t in trig[nm]]).to_csv(outd/f"stalta_triggers_{D_STATION_FILE[:6]}_{label}_{nm}.csv", index=False)
    print(f"STA/LTA triggers: raw {len(trig['raw'])} vs separated {len(trig['separated'])}")
    nh = int(np.ceil(len(raw) / f_s / 3600)); edges_h = np.arange(nh + 1)
    fig, ax = plt.subplots(figsize=(9, 4))
    ax.hist([[t["time"] - t_start for t in trig["separated"]], [t["time"] - t_start for t in trig["raw"]]], bins=edges_h * 3600, label=["separated earthquake", "raw"])
    ax.plot((edges_h[:-1] + .5) * 3600, np.histogram(ev_t[:, 0], bins=edges_h * 3600)[0], "ko-", label="catalogue M>=2")
    ax.set_xlabel("seconds since start"); ax.set_ylabel("triggers per hour"); ax.legend(); save_fig(fig, f"partD_stalta_{D_STATION_FILE[:6]}_{label}"); plt.close(fig)
    json.dump(dict(model=label, file=D_STATION_FILE, preprocess=D_PREPROCESS, params=P, triggers_raw=len(trig["raw"]), triggers_separated=len(trig["separated"]),
                   catalogue_events_m2=int(len(ev_t))), open(outd/f"partD_summary_{label}.json", "w"), indent=2)
    log_entry("D", f"real data {D_STATION_FILE} with {label}", f"triggers raw {len(trig['raw'])}, separated {len(trig['separated'])}")
    print("step7b (cross-correlation) is NOT implemented: step7b stacks one month (44,760 windows) and cannot run on the 1-day example files without changing its logic. The README lists it; whether the paper reports it still has to be checked.")
else:
    print("Part D skipped")


Part D skipped


## Part E: scorecard, deviation log, per-run config files (Phase 1.5)
Switch `RUN_PART_E`. Columns paper figure/table, paper value (unless entered in `PAPER_VALUES`), match and reason are left empty for the team.

In [15]:
def fmt(m, s):
    return "" if pd.isna(m) else (f"{m:.4f}" if pd.isna(s) else f"{m:.4f} +/- {s:.4f}")

def build_scorecard():
    A = read_csv_keep(RESULTS/"table_ev_per_bottleneck.csv"); o = A[A.kind == "ours"].set_index("bottleneck"); pr = A[A.kind == "pretrained"].set_index("bottleneck")
    tab = read_csv_keep(RESULTS/"table_ev_vs_snr.csv"); facts = json.load(open(RESULTS/"data_facts.json")); tf = json.load(open(RESULTS/"test_set_facts.json"))
    leak = json.load(open(RESULTS/"leakage_audit.json")); rows = []
    def add(claim, ours, paper=""): rows.append(dict(claim=claim, paper_figure_table="", paper_value=paper, our_value=ours, match="", reason=""))
    add("C1: training set size", f"{facts['n_rows']} rows ({facts['n_poha']} POHA + {facts['n_stead']} STEAD)")
    add("C1: unique source draws in the final file", f"{facts['n_unique_source_index']} ({facts['pct_unique']:.1f} %)")
    add("C1: test mixtures with <1 % earthquake energy in the window", f"{100 * tf['frac_energy_lt_1pct']:.2f} %")
    for b in BOTTLENECKS:
        if b not in o.index: add(f"C2: {b}", "no finished run"); continue
        pv = lambda k: PAPER_VALUES[k].get(b)
        add(f"C2: median EV earthquake, {b}", fmt(o.loc[b, "median_ev_eq_mean"], o.loc[b, "median_ev_eq_std"]), "" if pv("ev_earthquake") is None else pv("ev_earthquake"))
        add(f"C2: median EV noise, {b}", fmt(o.loc[b, "median_ev_noise_mean"], o.loc[b, "median_ev_noise_std"]), "" if pv("ev_noise") is None else pv("ev_noise"))
        add(f"C2: test loss, {b}", fmt(o.loc[b, "test_loss_mean"], o.loc[b, "test_loss_std"]), "" if pv("test_loss") is None else pv("test_loss"))
        add(f"Table 2: parameters, {b}", str(o.loc[b, "n_params"]), "" if pv("n_params") is None else pv("n_params"))
        if b in pr.index:
            add(f"C3: ours minus authors' pretrained, median EV earthquake, {b}", f"{o.loc[b, 'median_ev_eq_mean'] - pr.loc[b, 'median_ev_eq_mean']:+.4f} (pretrained {pr.loc[b, 'median_ev_eq_mean']:.4f}; external reference)")
    rk = o.sort_values("median_ev_eq_mean", ascending=False).index.tolist()
    add("C6: bottleneck ranking by median EV earthquake (best first)", " > ".join(rk))
    for b in BOTTLENECKS:
        t = tab[(tab.curve == f"ours_{b}_seedmean") & (tab.group == "all") & (tab.ev_kind == "eq") & (tab.n > 1)].sort_values("bin_lo")
        if len(t): add(f"C7: mean clipped EV earthquake, lowest vs highest SNR bin, {b}", f"{t.mean_ev_clipped.iloc[0]:.3f} (SNR {t.bin_lo.iloc[0]:.2g}-{t.bin_hi.iloc[0]:.2g}) vs {t.mean_ev_clipped.iloc[-1]:.3f} (SNR {t.bin_lo.iloc[-1]:.2g}-{t.bin_hi.iloc[-1]:.2g})")
    fp = RESULTS/"failure_cases_summary.json"
    if fp.exists():
        fs = json.load(open(fp)); add("C8: worst 2 % of test samples: share with <1 % earthquake energy", f"{100 * fs['frac_energy_lt_1pct']:.1f} % (model {fs['model']}; whole test set {100 * fs['test_frac_energy_lt_1pct']:.1f} %)")
    add("Leakage: test rows whose source_index also occurs in training", f"{100 * leak['frac_test_rows_sharing_source_index_with_train']:.1f} %")
    add("Leakage: test rows whose earthquake trace also occurs in training", f"{100 * leak['frac_test_rows_sharing_earthquake_name_with_train']:.1f} %")
    df = pd.DataFrame(rows); df.to_csv(RESULTS/"replication_scorecard.csv", index=False); return df

def write_deviation_log():
    reg = read_csv_keep(RESULTS/"run_table.csv") if (RESULTS/"run_table.csv").exists() else pd.DataFrame()
    gpus = sorted({str(g) for g in reg.get("gpu", []) if str(g) not in ("nan", "None", "")}); torchs = sorted({str(x) for x in pd.DataFrame(load_registry()).T.get("torch", []) if str(x) not in ("nan", "None")})
    L = ["# Deviation log (Phase 1)", "", "## From PATCH_NOTES.md",
         "1. Training data rebuilt from our own STEAD subset (custom download script); merged.csv row order unverified against the author's file, so the seeded draws may select different traces.",
         "2. step2c drops the last N mod 1000 windows (87,191 -> 87,000 written). Kept for fidelity.",
         "3. step2d shuffles with replacement and no seed (about 63 % unique). Kept, seeded; a permuted variant exists for the leakage experiment.",
         "4. README says the continuous noise is phase-shuffled; the code has shuffle_phase = False.",
         "5. step2b and step2c normalise differently.", "6. step2a in the repo does not produce the file step2c reads; replaced.",
         "7. Newer PyTorch / NumPy / Python than the repo pins (torch==1.10.2, scipy==1.8.0, Python 3.9).",
         "8. No early stopping (patience=None): always 100 epochs, last-epoch model saved.", "9. np.Inf -> np.inf in torch_tools.py.",
         "", "## Found while running Phase 1",
         f"- Repo commit {GIT_HASH}; embedded scripts sha256[:16]: {SCRIPT_SHA}.", f"- GPUs used for training: {gpus or 'not recorded'}; torch versions in run info: {torchs or 'not recorded'}.",
         "- Training seeds: " + ", ".join(map(str, PLANNED_SEEDS)) + " (torch seed only; split seeds 13 and 20 are fixed).",
         "- float64 training is kept for fidelity; cuDNN nondeterminism remains (cudnn.benchmark=False only).",
         "- EV is evaluated with step5's definition (sklearn explained_variance_score, one value per trace-component). Mean EV is dominated by a few extreme negative values (the authors' own stored arrays contain values near -1e58), so tables report the pooled median (as in step5's figures) plus a mean clipped at " + str(EV_CLIP) + " (our choice).",
         "- step5 drops the last incomplete batch of 256 (37,376 of 37,400 test samples); we evaluate all 37,400 and also report the step5-subset numbers.",
         "- Evaluation runs on the GPU with the torch eval fast path disabled; step5 ran on CPU with torch 1.10.",
         f"- Compatibility shim for old pickles used for: {COMPAT_LOG or 'nothing'}.",
         f"- Band-pass baseline band {BANDPASS_HZ} Hz is our choice (see configuration cell); confirm against the paper.",
         "- Confidence intervals cover sampling of test samples only; seed spread is reported separately.",
         "- Pretrained models were trained on a split we cannot see (the _Dataset_split.hdf5 files hold only attributes): external reference.",
         "- step7b not run (needs one month of data and a changed batch logic)."]
    (RESULTS/"deviation_log.md").write_text("\n".join(L) + "\n")

if RUN_PART_E:
    for run in discover_runs(EPOCHS_REQUIRED).values(): write_run_config(run)
    update_run_table()
    sc = build_scorecard(); write_deviation_log()
    print(sc.to_string(max_colwidth=90)); print("\nPaper value, match and reason are left for the team. Files: results/replication_scorecard.csv, results/deviation_log.md")
    log_entry("E", "scorecard + deviation log + run configs", f"{len(sc)} scorecard rows")
else:
    print("Part E skipped")


Part E skipped


**Reminders**
- One bottleneck and one seed per Part B session. The 15 runs (5 bottlenecks x 3 seeds) take many sessions; check the smoke projection first.
- The authors' pretrained models are an external reference. They were trained on a split we cannot see.
- Mean EV is dominated by extreme outliers; use the medians (as step5 does) and the clipped means.
- `step7b` is not implemented here; see Part D output.
